In [2]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
# Load and preprocess the data
df = pd.read_csv('dataset.csv')
df.dropna(subset=['Description', 'Class'], inplace=True)

C:\Users\Prasanna\AppData\Local\Temp\ipykernel_16312\1636880427.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('dataset.csv')


In [4]:
df

,Term ID,Class,Description,Status,Effective Date,Type,Notes,TM5,NCL Version
0,010-1886,10,Medical imaging apparatus for use in the field...,A,04/16/2020,GOODS,NaN,NaN,"""11-2020"""
1,010-1889,10,Medical diagnostic apparatus for testing {spec...,A,04/16/2020,GOODS,NaN,NaN,"""11-2020"""
2,028-4345,28,Balancing bird toys,A,04/16/2020,GOODS,Balancing bird toys are bird-shaped educationa...,T,"""11-2020"""
3,009-5776,9,Image intensifier tubes,A,04/16/2020,GOODS,Image intensifier tubes are a type of vacuum t...,T,"""11-2020"""
4,009-5779,9,Sunglasses for pets,A,04/16/2020,GOODS,Sunglasses is acceptable wording; further spec...,T,"""11-2020"""
...,...,...,...,...,...,...,...,...,...
68055,004-828,4,Fruitwood lump charcoal,A,08-08-2024,GOODS,"Lump charcoal, also known as natural hardwood ...",NaN,"""12-2024"""
68056,004-829,4,Hydrocarbon gas liquids (HGL) for use as fuel ...,A,08-08-2024,GOODS,NaN,NaN,"""12-2024"""
68057,004-830,4,Liquefied hydrocarbon gas (LHG) for use as fue...,A,08-08-2024,GOODS,NaN,NaN,"""12-2024"""
68058,004-831,4,Natural hardwood charcoal,A,08-08-2024,GOODS,"Natural hardwood charcoal is carbonized wood, ...",NaN,"""12-2024"""


In [5]:
# Drop rows where 'Class' is 'A', '200', or 'B'
values_to_drop = ['A', '200', 'B']
df = df[~df['Class'].isin(values_to_drop)]

In [6]:
X = df['Description']
y = df['Class']

In [7]:
# Text preprocessing
voc_size = 5000
ps = PorterStemmer()
corpus = []

In [8]:
for i in range(len(X)):
    review = re.sub('[^a-zA-Z]', ' ', X.iloc[i])
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [9]:
corpus

['medic imag apparatu use field indic e g iridolog sclerolog etc incorpor record softwar indic function purpos softwar',
 'medic diagnost apparatu test specifi condit subject matter test e g blood sugar level cancer cell dna etc incorpor record oper system softwar',
 'balanc bird toy',
 'imag intensifi tube',
 'sunglass pet',
 'water temperatur gaug',
 'exercis pulley',
 'automobil engin model toy',
 'remot control submarin toy',
 'magnet clip sunglass lens',
 'pork jerki',
 'medic devic use treat diagnos specifi diseas condit integr record oper system softwar sold unit',
 'softwar use calcul transfer store data relat diagnosi treatment eye disord sold compon ophthalm surgic apparatu',
 'medic imag apparatu incorpor medic imag softwar',
 'medic imag apparatu indic medic use e g diagnos medic condit use surgic procedur etc incorpor record softwar indic function purpos softwar',
 'medic imag apparatu indic medic use e g diagnos medic condit use surgic procedur etc incorpor record oper sy

In [21]:
# One-hot encode the corpus
onehot_repr = [one_hot(words, voc_size) for words in corpus]

In [22]:
# Pad the sequences
sent_length = 20
embedded_docs = pad_sequences(onehot_repr, padding='pre', maxlen=sent_length)

In [23]:
df['Class'] = df['Class'].astype(str)

C:\Users\Prasanna\AppData\Local\Temp\ipykernel_16312\1111352407.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Class'] = df['Class'].astype(str)


In [24]:
# Encode labels as integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Class'])

In [25]:
# One-hot encode the labels
y = to_categorical(y, num_classes=len(np.unique(y)))

In [26]:
# Convert to numpy arrays
X_final = np.array(embedded_docs)
y_final = np.array(y)


In [27]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)


In [28]:
# Build the model function for Keras Tuner
def build_model(hp):
    model = Sequential()
    model.add(Embedding(voc_size, hp.Int('embedding_output_dim', min_value=30, max_value=100, step=10), input_length=sent_length))
    model.add(LSTM(hp.Int('lstm_units', min_value=50, max_value=200, step=50)))
    model.add(Dense(46, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [29]:
# Initialize the Hyperband tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='my_dir',
    project_name='text_classification'
)

Reloading Tuner from my_dir\text_classification\tuner0.json


In [30]:
# Define early stopping
stop_early = EarlyStopping(monitor='val_loss', patience=5)


In [31]:

# Perform hyperparameter search
tuner.search(X_train, y_train, epochs=50, validation_split=0.2, callbacks=[stop_early])


In [32]:

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

In [33]:
# Build the model with the best hyperparameters
model = tuner.hypermodel.build(best_hps)

In [34]:
# Train the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10)



Epoch 1/10


1425/1425 [==============================] - 61s 39ms/step - loss: 1.9422 - accuracy: 0.4786 - val_loss: 1.1894 - val_accuracy: 0.6661
Epoch 2/10
1425/1425 [==============================] - 53s 37ms/step - loss: 0.9206 - accuracy: 0.7388 - val_loss: 0.9872 - val_accuracy: 0.7221
Epoch 3/10
1425/1425 [==============================] - 52s 37ms/step - loss: 0.6621 - accuracy: 0.8046 - val_loss: 0.9152 - val_accuracy: 0.7394
Epoch 4/10
1425/1425 [==============================] - 53s 37ms/step - loss: 0.5138 - accuracy: 0.8438 - val_loss: 0.9068 - val_accuracy: 0.7503
Epoch 5/10
1425/1425 [==============================] - 53s 37ms/step - loss: 0.4126 - accuracy: 0.8719 - val_loss: 0.9366 - val_accuracy: 0.7498
Epoch 6/10
1425/1425 [==============================] - 53s 37ms/step - loss: 0.3348 - accuracy: 0.8939 - val_loss: 0.9714 - val_accuracy: 0.7538
Epoch 7/10
1425/1425 [==============================] - 53s 37ms/step - loss: 0.2763 - accuracy: 0.9116 - val_loss: 0.9919

In [35]:
model.save('model.h5')

c:\Users\Prasanna\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [44]:
import pickle

# Save the LabelEncoder
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)


In [53]:
# Save the PorterStemmer (if applicable)
with open('porter_stemmer.pkl', 'wb') as file:
    pickle.dump(ps, file)

# Save any important preprocessing variables
preprocessing_info = {
    'voc_size': voc_size,
    'sent_length': sent_length
}

with open('preprocessing_info.pkl', 'wb') as file:
    pickle.dump(preprocessing_info, file)


# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test accuracy: {accuracy:.2f}")

# Print out the best hyperparameters
print(f"Best Hyperparameters: {best_hps.values}")


In [37]:
# Function to predict class for a new description
def predict_class(description, model, ps, label_encoder, voc_size, sent_length):
    # Preprocess the input description
    review = re.sub('[^a-zA-Z]', ' ', description)
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    
    # One-hot encode the processed description
    onehot_repr = one_hot(review, voc_size)
    embedded_doc = pad_sequences([onehot_repr], padding='pre', maxlen=sent_length)
    
    # Predict class probabilities
    prediction = model.predict(embedded_doc)
    
    # Get the class with the highest probability
    predicted_class_index = np.argmax(prediction, axis=1)[0]
    predicted_class = label_encoder.inverse_transform([predicted_class_index])[0]
    
    return predicted_class

In [38]:
# Test the model with a new description
sample_description = "Providing temporary use of on-line non downloadable software"
predicted_class = predict_class(sample_description, model, ps, label_encoder, voc_size, sent_length)
print(f"The predicted class for the given description is: {predicted_class}")

1/1 [==============================] - 1s 870ms/step
The predicted class for the given description is: 42


REST API